# PRÁCTICA OBLIGATORIA: Clasificación y Regresión Logística
Modelo de supervivencia en el Titanic + parte extra multiclase.

**Mapa del notebook** (cada bloque coincide con un paso del enunciado):
0. Imports · 1. Carga · 2. Limpieza · 3. Target · 4. Split · 5. X, y · 6. Imputación · 7. Mini-EDA · 8. Tratamiento de variables · 9. Modelo · 10. Pesos · 11. Procesar test · 12. Evaluación · 13. Probabilidad extra · 14. Multiclase

## Ejercicio 0 · Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, roc_curve, roc_auc_score)

## 1. Carga del dataset

In [ ]:
try:
    df = pd.read_csv("../data/titanic.csv")
except FileNotFoundError:
    # Plan B por si no tienes la carpeta data: seaborn trae el mismo dataset
    df = sns.load_dataset("titanic")

df.head()

In [ ]:
df.info()

In [ ]:
df.deck.value_counts(dropna=False, normalize=True)

In [ ]:
df.nunique() / len(df) * 100

## 2. Limpieza (sin imputar)
Quito variables que **no voy a usar** y explico por qué:

| Variable | Motivo |
|---|---|
| `alive` | Es el target escrito con otras palabras (*yes/no*). Usarla sería **hacer trampa** (data leakage). |
| `class` | Es idéntica a `pclass`, solo cambia el formato. |
| `embark_town` | Es idéntica a `embarked` (S/C/Q ↔ ciudad). |
| `who`, `adult_male` | Se derivan de `sex` y `age`; no aportan información nueva. |
| `deck` | ~77 % de nulos: no hay datos suficientes para que sea fiable. |

In [ ]:
cols_fuera = ["alive", "class", "embark_town", "who", "adult_male", "deck"]
df = df.drop(columns=[c for c in cols_fuera if c in df.columns])
df.isna().sum()

## 3. Variable target: `survived`
Queremos predecir si un pasajero **sobrevivirá** (1) o no (0).

In [ ]:
print(df["survived"].value_counts(normalize=True).round(3))

fig, ax = plt.subplots(figsize=(4, 3))
sns.countplot(data=df, x="survived", ax=ax)
ax.set_xticklabels(["No sobrevive (0)", "Sobrevive (1)"])
ax.set_title("Distribución del target")
plt.show()

**Razonamiento:** aproximadamente el 38 % sobrevive y el 62 % no. Está **algo desbalanceado**, aunque no de forma extrema.
- Un modelo que dijera siempre "no sobrevive" ya acertaría ~62 %, así que la **accuracy sola engaña**.
- Por eso miro también *precision, recall, F1* de la clase 1, la matriz de confusión y la curva ROC/AUC.
- Es esperable que el modelo acierte algo mejor con los "no sobreviven" (clase mayoritaria) que con los "sobreviven".

## 4. Split train/test
Estratifico por `embarked` para que las tres ciudades (S, C, Q) estén repartidas de forma similar en ambos conjuntos.
`embarked` tiene 2 nulos, así que los marco temporalmente como `"missing"` **solo para estratificar** (no es una imputación del dato).

In [ ]:
df_train, df_test = train_test_split(
    df, test_size=0.2, random_state=42,
    stratify=df["embarked"].fillna("missing"))

# Comprobación: los repartos deben ser parecidos
pd.concat([df_train["embarked"].value_counts(normalize=True).rename("train"),
           df_test["embarked"].value_counts(normalize=True).rename("test")], axis=1).round(3)

## 5. Parejas X, y

In [ ]:
X_train = df_train.drop(columns="survived")
y_train = df_train["survived"]
X_test  = df_test.drop(columns="survived")
y_test  = df_test["survived"]
X_train.shape, X_test.shape

## 6. Imputación (solo con datos de train)
Para evitar *data leakage*, calculo medias y modas **solo con train** y las guardo para reutilizarlas en test.
- Numéricas (`age`, `fare`) → **media**
- Categóricas (`embarked`, `sex`, `pclass`) → **moda**

En la sección 8 meto todo esto dentro de una función para aplicarla igual a train y test.

In [ ]:
medias = X_train[["age", "fare"]].mean()
modas  = {"embarked": X_train["embarked"].mode()[0],
          "sex":      X_train["sex"].mode()[0],
          "pclass":   X_train["pclass"].mode()[0]}
print(medias, "\n", modas)

## 7. Mini-EDA y selección de features

In [ ]:
eda = X_train.assign(survived=y_train)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
sns.barplot(data=eda, x="sex", y="survived", ax=axes[0, 0]);      axes[0, 0].set_title("Supervivencia por sexo")
sns.barplot(data=eda, x="pclass", y="survived", ax=axes[0, 1]);   axes[0, 1].set_title("Supervivencia por clase")
sns.barplot(data=eda, x="embarked", y="survived", ax=axes[0, 2]); axes[0, 2].set_title("Supervivencia por embarque")
sns.histplot(data=eda, x="age", hue="survived", bins=25, ax=axes[1, 0]); axes[1, 0].set_title("Edad")
sns.boxplot(data=eda, x="survived", y="fare", ax=axes[1, 1]);     axes[1, 1].set_title("Tarifa")
sns.barplot(data=eda, x="alone", y="survived", ax=axes[1, 2]);    axes[1, 2].set_title("Viaja solo")
plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
sns.heatmap(eda.select_dtypes("number").corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlaciones (train)"); plt.show()

**Conclusiones del EDA (lo que suele verse en el Titanic):**
- `sex`: la diferencia más grande; las mujeres sobreviven mucho más.
- `pclass` y `fare`: a mejor clase / mayor tarifa, más supervivencia (y entre ellas están correlacionadas).
- `age`: los niños pequeños sobreviven algo más.
- `embarked`: efecto menor (Cherburgo algo mejor, probablemente porque iba más pasaje de 1ª clase).
- `alone`, `sibsp`, `parch`: efecto pequeño; `alone` resume bastante bien a las otras dos.

**Features que me quedo:** `pclass, sex, age, fare, embarked, alone`.
Descarto `sibsp` y `parch` porque `alone` ya recoge lo esencial y así evito redundancia.

In [ ]:
features = ["pclass", "sex", "age", "fare", "embarked", "alone"]
X_train = X_train[features]
X_test  = X_test[features]

## 8. Tratamiento de variables
Dos funciones:
- `ajustar_prep(X)`: **aprende** de train (medias, modas, escalado, columnas finales).
- `aplicar_prep(X, prep)`: **aplica** lo aprendido a cualquier dataset (train, test, un pasajero inventado…).

Qué hace:
1. Imputa nulos (medias/modas de train).
2. `alone` → 0/1.
3. Escala `age` y `fare` (StandardScaler) para que la regresión logística trabaje bien.
4. One-hot de `sex`, `embarked` y `pclass` (quitando una categoría base para no duplicar información).

In [ ]:
def aplicar_prep(X, prep):
    X = X.copy()
    num, cat = prep["num"], prep["cat"]
    X[num] = X[num].fillna(prep["medias"])
    for c in cat:
        X[c] = X[c].fillna(prep["modas"][c]).astype(str)
    if "alone" in X:
        X["alone"] = X["alone"].astype(int)
    X[num] = prep["scaler"].transform(X[num])
    if prep.get("columnas") is None:          # fase de ajuste (train)
        X = pd.get_dummies(X, columns=cat, drop_first=True)
    else:                                     # fase de aplicación (test, etc.)
        X = pd.get_dummies(X, columns=cat).reindex(columns=prep["columnas"], fill_value=0)
    return X.astype(float)


def ajustar_prep(X):
    prep = {}
    prep["num"] = [c for c in ["age", "fare"] if c in X]
    prep["cat"] = [c for c in ["sex", "embarked", "pclass"] if c in X]
    prep["medias"] = X[prep["num"]].mean()
    prep["modas"] = {c: X[c].mode()[0] for c in prep["cat"]}
    prep["scaler"] = StandardScaler().fit(X[prep["num"]].fillna(prep["medias"]))
    prep["columnas"] = None
    prep["columnas"] = aplicar_prep(X, prep).columns   # guarda las columnas finales
    return prep


prep = ajustar_prep(X_train)
X_train_p = aplicar_prep(X_train, prep)
X_train_p.head()

## 9. Modelo de regresión logística

In [ ]:
modelo = LogisticRegression(max_iter=1000)
modelo.fit(X_train_p, y_train)

## 10. (EXTRA) Pesos e importancia de las features
Como las numéricas están escaladas, los coeficientes son comparables.
- Coeficiente positivo → sube la probabilidad de sobrevivir. Negativo → la baja.
- `exp(coef)` = *odds ratio*: cuánto se multiplican las "probabilidades relativas" (odds).

In [ ]:
pesos = pd.DataFrame({"coef": modelo.coef_[0],
                      "odds_ratio": np.exp(modelo.coef_[0])},
                     index=X_train_p.columns).sort_values("coef")
display(pesos.round(3))

pesos["coef"].plot(kind="barh", figsize=(6, 4), title="Peso de cada feature")
plt.axvline(0, color="k", lw=0.8); plt.show()

## 11. Procesar el test (con lo aprendido en train)

In [ ]:
X_test_p = aplicar_prep(X_test, prep)
X_test_p.shape

## 12. Evaluación en train y test

In [ ]:
y_pred_train = modelo.predict(X_train_p)
y_pred_test  = modelo.predict(X_test_p)

print("===== TRAIN =====")
print(classification_report(y_train, y_pred_train))
print("===== TEST =====")
print(classification_report(y_test, y_pred_test))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, y_real, y_hat, nombre in [(axes[0], y_train, y_pred_train, "Train"),
                                  (axes[1], y_test,  y_pred_test,  "Test")]:
    ConfusionMatrixDisplay(confusion_matrix(y_real, y_hat),
                           display_labels=["No sobrevive", "Sobrevive"]).plot(ax=ax, colorbar=False)
    ax.set_title(f"Matriz de confusión · {nombre}")
plt.tight_layout(); plt.show()

In [ ]:
proba_train = modelo.predict_proba(X_train_p)[:, 1]
proba_test  = modelo.predict_proba(X_test_p)[:, 1]

plt.figure(figsize=(5, 5))
for y_real, proba, nombre in [(y_train, proba_train, "Train"), (y_test, proba_test, "Test")]:
    fpr, tpr, _ = roc_curve(y_real, proba)
    plt.plot(fpr, tpr, label=f"{nombre} (AUC = {roc_auc_score(y_real, proba):.3f})")
plt.plot([0, 1], [0, 1], "k--", label="Azar")
plt.xlabel("Tasa de falsos positivos"); plt.ylabel("Tasa de verdaderos positivos")
plt.title("Curva ROC"); plt.legend(); plt.show()

In [ ]:
# Comparación de predicciones con valores reales (primeras filas de test)
comparacion = pd.DataFrame({"real": y_test.values,
                            "predicho": y_pred_test,
                            "prob_sobrevivir": proba_test.round(3)}, index=y_test.index)
comparacion["acierto"] = comparacion["real"] == comparacion["predicho"]
comparacion.head(10)

**Valoración de la generalización (rellena con tus números al ejecutar):**
- Compara accuracy, F1 y AUC de train vs test. Si son **parecidos** (diferencia de unos pocos puntos), el modelo **generaliza bien**: no hay sobreajuste.
- Si train fuera mucho mejor que test, habría sobreajuste. Con un modelo lineal y pocas variables, lo normal es que esté bien.
- Fíjate en el *recall* de la clase 1: indica qué porcentaje de supervivientes reales detecta el modelo.
- Es normal que el test varíe un poco respecto a train porque es pequeño (~180 pasajeros).

## 13. (EXTRA) Probabilidad de supervivencia de un perfil concreto
Hombre, entre 30 y 40 años, embarcó en Southampton (S), 2ª clase, viaja solo, billete = 0.

Como la edad es un rango, calculo la probabilidad para **cada edad entera de 30 a 40** y hago la **media**.

In [ ]:
edades = np.arange(30, 41)
perfil = pd.DataFrame({
    "pclass":   2,
    "sex":      "male",
    "age":      edades,
    "fare":     0.0,
    "embarked": "S",
    "alone":    True,
})[features]

probas = modelo.predict_proba(aplicar_prep(perfil, prep))[:, 1]
print(pd.Series(probas.round(3), index=edades))
print(f"\nProbabilidad media de supervivencia: {probas.mean():.3f}  ({probas.mean()*100:.1f} %)")

## 14. (EXTRA) Multiclase: predecir la clase del viajero (`pclass`)
Único cambio: el **target pasa a ser `pclass`** y `survived` entra como feature.
Reutilizo exactamente las mismas funciones de procesado.

In [ ]:
# Nuevo X: quito pclass y añado survived
X_train_m = X_train.drop(columns="pclass").assign(survived=y_train)
X_test_m  = X_test.drop(columns="pclass").assign(survived=y_test)
y_train_m = X_train["pclass"]
y_test_m  = X_test["pclass"]

prep_m = ajustar_prep(X_train_m)
X_train_mp = aplicar_prep(X_train_m, prep_m)
X_test_mp  = aplicar_prep(X_test_m,  prep_m)

modelo_m = LogisticRegression(max_iter=1000)   # multinomial automáticamente con 3 clases
modelo_m.fit(X_train_mp, y_train_m)

print(classification_report(y_test_m, modelo_m.predict(X_test_mp)))

**Comentario:** ahora hay 3 clases (1ª, 2ª, 3ª). Suele ir muy bien con la 3ª y la 1ª (por `fare`) y peor con la 2ª, que se parece a ambas. Mira el *recall* y el *F1* de cada clase para comprobarlo.